# Etapa 5k — Wiener con el ruido estimado en cada imagen

**Ronda de mejoras para la entrega final (desvío 14.11 del plan).** Solo desarrollo, la misma validación cruzada por personas. El conjunto de prueba no se usa.

**Motivo.** En la etapa 8 el algoritmo funcionó hasta ruido de desvío 5 niveles. Con 10 Matthews cae 0,07 y con 20 cae a 0,41. El Wiener local aplicado a la foto lo recuperaba (+0,05 con σ = 10, +0,24 con σ = 20), pero no se agregó al algoritmo porque en la base no cambiaba nada; quedó como recomendación aplicarlo a fotos con ruido marcado (desvío 14.9). Para que el algoritmo lo haga solo, hace falta saber cuánto ruido tiene cada foto. El Wiener de scipy estima el ruido con el promedio de las varianzas locales: en una foto limpia ese promedio sale alto por los propios vasos, así que la suaviza igual.

**Qué se propone** (`preprocess.wiener_ruido_estimado`, paso "Wiener con ruido estimado"). Sobre el verde a la mitad de resolución, antes del relleno de exudados y de la normalización:
1. **Estimar el desvío del ruido** (`preprocess.estimar_ruido`). Se filtra con la máscara de Immerkær (1996), dos laplacianos restados que anulan el fondo suave y dejan pasar casi solo el ruido, y se toma la mediana del valor absoluto: la desviación mediana absoluta de Donoho y Johnstone (1994), para que los bordes de los vasos no inflen la estimación. En desarrollo estima 0,5 a 0,7 niveles en las fotos limpias, y 2,47, 4,94 y 9,88 con ruido agregado de 5, 10 y 20 (la reducción 2×2 baja el desvío a la mitad: lo esperado es 2,5, 5 y 10).
2. **Wiener local** (Lee, 1980) en ventanas de 5 × 5 (la de la etapa 3), con ese ruido: media + (1 − ruido / varianza local) × (valor − media). En una foto limpia casi no cambia nada (0,1 a 0,2 niveles en promedio, contra 0,9 a 1,1 con el ruido de scipy); en una ruidosa, suaviza el fondo y respeta los vasos.

**Candidatas** (sobre el algoritmo actual: división + relleno de exudados, 28 mapas, entrenamiento equilibrado por grosor, valor fijo elegido en entrenamiento). Cada una ajusta sus pesos y su valor fijo con las fotos de entrenamiento sin ruido agregado (la base), como en la etapa 8:
- **A.** Sin Wiener (el algoritmo actual).
- **B.** Con Wiener y el ruido estimado.
- **C.** Con el Wiener de la etapa 3 (`restoration.wiener_denoise`, ruido de scipy). Es descriptiva: muestra qué pasa si se filtra siempre. No se puede adoptar.

**Condiciones.** Las fotos de las personas de afuera tal como están, y con ruido gaussiano agregado de desvío 5, 10 y 20 niveles (`degrade.gaussian_noise`, el mismo ruido para las tres candidatas).

**Medidas**, a resolución original, en el campo de visión de la base: Matthews, Dice de línea central, vasos finos (sensibilidad de línea central) y precisión. Descriptivo: el ruido estimado en cada condición.

**Regla de adopción (fijada antes de correr).** B reemplaza a A si
1. en las fotos sin ruido agregado no es inferior a A en Matthews, vasos finos ni Dice de línea central (límite inferior > −0,03), **y**
2. con ruido de desvío 10 (el primer nivel en que el algoritmo dejó de funcionar en la etapa 8) mejora Matthews con un intervalo al 95 % que no incluye el cero.

Si no cumple, queda A.

**Control:** A sin ruido agregado tiene que reproducir el Matthews de A en `05j` (0,7533).

In [ ]:
import sys
import time
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from src import config as C
from src.data import dev_ids, load_split
from src.evaluation import map_images
from src.pipeline import choose_value, fit_weights
from src.preprocess import PASO_WIENER_ESTIMADO, PASO_WIENER_LOCAL
from src.split import person_folds
from src.stats import percentile_interval, person_bootstrap_means

pd.set_option("display.width", 220)
ids = dev_ids()
dev = load_split().set_index("image").loc[ids]
folds = dev["person"].map(person_folds(dev.reset_index(), 5, C.SEED))
CANDS = {"A. sin Wiener": C.SEGMENT_BACKGROUND_MODE,
         "B. Wiener con ruido estimado": C.SEGMENT_BACKGROUND_MODE + " + " + PASO_WIENER_ESTIMADO,
         "C. Wiener de la etapa 3": C.SEGMENT_BACKGROUND_MODE + " + " + PASO_WIENER_LOCAL}
ORDER = list(CANDS)
NOISE = (0, 5, 10, 20)
WORKERS = 4
print(CANDS, "|", len(C.SEGMENT_FEATURES), "mapas")

## Pasada 1: píxeles de entrenamiento (fotos sin ruido agregado)

In [ ]:
def pass1(image_id):
    from src.pipeline import training_pixels
    return image_id, {c: training_pixels(image_id, background=m) for c, m in CANDS.items()}

t = time.time()
P1 = dict(map_images(pass1, ids, workers=WORKERS))
samples = {c: {i: P1[i][c] for i in ids} for c in CANDS}
del P1
print(f"Pasada 1: {time.time() - t:.0f} s")

## Ajuste por vuelta

In [ ]:
fit = {}
for k in range(5):
    tr = list(folds.index[folds != k])
    for c in CANDS:
        w = fit_weights(samples[c], tr)
        fit[(k, c)] = (w, choose_value(samples[c], w, tr))
pd.DataFrame({c: [fit[(k, c)][1] for k in range(5)] for c in CANDS}, index=[f"vuelta {k}" for k in range(5)]).round(4)

## Pasada 2: las personas de afuera, sin ruido agregado y con ruido 5, 10 y 20

In [ ]:
def pass2(image_id):
    import numpy as np
    from src.config import WIDTH_LIMITS_PX
    from src.data import load_fov, load_image, load_vessels
    from src.degrade import gaussian_noise
    from src.enhance import prepare_maps, upscale
    from src.fov import detect_fov_robust
    from src.metrics import centerline_dice, matthews
    from src.pipeline import feature_scores
    from src.preprocess import estimar_ruido
    from src.vessels import skeleton_widths
    k = int(folds[image_id])
    clean = load_image(image_id)
    truth, fov = load_vessels(image_id), load_fov(image_id)
    skel, wd = skeleton_widths(truth & fov)
    thin = skel & (wd <= WIDTH_LIMITS_PX[0])
    rows = []
    for s in NOISE:
        rgb = clean if s == 0 else gaussian_noise(clean, s, C.SEED + 1000 * s + ids.index(image_id))
        maps, region, _ = prepare_maps(rgb, ["Verde"], 2)
        noise_est = 255 * estimar_ruido(maps["Verde"], region)
        del maps
        auto = detect_fov_robust(rgb)
        for c, mode in CANDS.items():
            w, value = fit[(k, c)]
            (p,), _, _ = feature_scores(rgb, [w], 2, background=mode)
            m = (upscale(p, fov.shape) > value) & auto
            marked = m & fov
            rows.append({"image": image_id, "ruido": s, "candidata": c, "ruido_estimado": noise_est,
                         "matthews": matthews(m, truth, fov), "dice_linea_central": centerline_dice(m, truth, fov),
                         "finos": m[thin].mean(), "precision": (marked & truth).sum() / max(marked.sum(), 1)})
    return rows

t = time.time()
R = pd.DataFrame([r for rows in map_images(pass2, ids, workers=WORKERS) for r in rows])
R.to_csv(ROOT / "results" / "05k_wiener.csv", index=False)
print(f"Pasada 2: {time.time() - t:.0f} s")

## Resultados

In [ ]:
print("Ruido estimado (niveles de gris, verde a la mitad de resolución; lo esperado es el agregado / 2):")
display(R[R["candidata"] == ORDER[0]].groupby("ruido")["ruido_estimado"].describe()[["mean", "min", "max"]].round(2))
print("Matthews medio por condición:")
R.pivot_table(index="candidata", columns="ruido", values="matthews").loc[ORDER].round(4)

In [ ]:
def compare(new, s):
    out = []
    sub = R[R["ruido"] == s]
    for meas in ("matthews", "dice_linea_central", "finos", "precision"):
        w = sub.pivot(index="image", columns="candidata", values=meas)
        d = (w[new] - w[ORDER[0]]).to_frame()
        lo, hi = percentile_interval(person_bootstrap_means(d, dev["person"]))
        out.append({"candidata": new[:2], "ruido": s, "medida": meas, "A": w[ORDER[0]].mean(), "nueva": w[new].mean(),
                    "diferencia": d.iloc[:, 0].mean(), "inferior": lo[0], "superior": hi[0],
                    "imágenes que mejoran": int((d.iloc[:, 0] > 0).sum())})
    return out

table = pd.DataFrame([r for c in ORDER[1:] for s in NOISE for r in compare(c, s)]).set_index(["candidata", "ruido", "medida"])
table.round(4)

In [ ]:
b = table.loc["B."]
cond1 = bool(all(b.loc[(0, m), "inferior"] > -C.NONINF_MARGIN_MATTHEWS
                 for m in ("matthews", "finos", "dice_linea_central")))
cond2 = bool(b.loc[(10, "matthews"), "inferior"] > 0)
print("B no inferior sin ruido agregado (1):", cond1, "| B mejora Matthews con ruido 10 (2):", cond2)
print("→ se adopta:", ORDER[1] if cond1 and cond2 else ORDER[0])

In [ ]:
print("Matthews por grupo, sin ruido agregado y con ruido 10:")
R[R["ruido"].isin([0, 10])].join(dev["group"], on="image").pivot_table(
    index="candidata", columns=["ruido", "group"], values="matthews").loc[ORDER].round(3)

## Decisión (escrita después de correr)

*(pendiente)*